# Primer paso cargar el API Key y modelos

En este primer paso cargamos las librerías necesarias para poder trabajar sobre nuestros modelos y el PDF. Utilizaremos los modelos de OPENAI cargando la API KEY desde el archivo de configuración.

In [ ]:
import os, getpass
import requests
from dotenv import load_dotenv
from pathlib import Path
from pypdf import PdfReader
from langchain_core.documents import Document
from langchain.chat_models import init_chat_model
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
from typing_extensions import List, TypedDict
from langchain_core.documents import Document
from langgraph.graph import START, StateGraph
from langchain_core.prompts import ChatPromptTemplate
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage
from typing import Annotated
from typing_extensions import List, TypedDict, Literal, Any
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, BaseMessage
from langgraph.graph.message import add_messages
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode, tools_condition

dotenv_loaded = load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key: ")

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("No se encontraron las API keys requeridas. Por favor, agrégalas al archivo .env o introdúcelas manualmente.")

GENERATION_MODEL = "gpt-5-mini"
EMBEDDING_MODEL = "text-embedding-ada-002"
PDF_PATH = Path("data/TENERIFE.pdf")

llm = init_chat_model(GENERATION_MODEL, model_provider="openai")
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)


# Recoger las secciones del documento

Aquí se esta cargando todo el documento para posteriormente coger todas las páginas y unificarlas en una sola página. De esta forma, podemos recorrer toda la página y coger la información por las diferentes secciones del documento sitios que ver y sitios para comer. También tengo que coger de que zona son los sitios para ver si se pregunta por norte o sur. Por último coger los lugarés de interés así que hago documentos separados por los lugarés de interés identificados por sección y zona.

In [46]:
if not PDF_PATH.exists():
    raise FileNotFoundError(f"No se encontró el PDF en {PDF_PATH.resolve()}")

reader = PdfReader(str(PDF_PATH))

document = "".join(
    [page.extract_text().strip() for page in reader.pages if page.extract_text()]
)

visited_places = [
    "• Santa Cruz de Tenerife",
    "• San Cristóbal de La Laguna",
    "• Parque rural de Anaga",
    "• La Orotava",
    "• Puerto de La Cruz",
    "• Isla Baja (Icod de los Vinos, Garachico y Punta de Teno)",
    "• El Teide",
    "• Costa Adeje",
    "• Los Gigantes",
    "• Masca",
    "• El Médano / La Tejita",
    "• Los Cristianos",
]

other_places = [
    "• Sitios para comer (sobre todo os podré decir de zona norte)"
]


north_zone = [
    "• Santa Cruz de Tenerife",
    "• San Cristóbal de La Laguna",
    "• Parque rural de Anaga",
    "• La Orotava",
    "• Puerto de La Cruz",
    "• Isla Baja (Icod de los Vinos, Garachico y Punta de Teno)",
    "• El Teide"
]

south_zone = [
    "• Costa Adeje",
    "• Los Gigantes",
    "• Masca",
    "• El Médano / La Tejita",
    "• Los Cristianos"
]

places = [
    "• Santa Cruz de Tenerife",
    "• San Cristóbal de La Laguna",
    "• Parque rural de Anaga",
    "• La Orotava",
    "• Puerto de La Cruz",
    "• Isla Baja (Icod de los Vinos, Garachico y Punta de Teno)",
    "• El Teide",
    "• Costa Adeje",
    "• Los Gigantes",
    "• Masca",
    "• El Médano / La Tejita",
    "• Los Cristianos",
    "• Sitios para comer (sobre todo os podré decir de zona norte)"
]

def split_places(text: str, splits: list) -> dict:
    places = {}
    other_text = text
    
    for i in range(len(splits)):
        actual = splits[i]
        next = splits[i + 1] if i + 1 < len(splits) else None
        
        init_idx = other_text.find(actual)
        if init_idx == -1:
            continue
            
        if next:
            find_idx = other_text.find(next, init_idx + len(actual))
            if find_idx != -1:
                section = other_text[init_idx:find_idx]
            else:
                section = other_text[init_idx:]
        else:
            section = other_text[init_idx:]
            
        places[actual] = section.strip()
        
    return places

places_dict = split_places(document, places)

docs = [] 

zones = [
    "ZONA NORTE", "ZONA SUR"
]

for place in places_dict.keys():
    for page_number, page in enumerate(reader.pages):
        if place in page.extract_text():
            section = "SITIOS QUE VER" if place in visited_places else "SITIOS PARA COMER" if place in other_places else place
            zone = "ZONA NORTE" if place in north_zone else "ZONA SUR" if place in south_zone else None
            doc = Document(
                page_content=places_dict[place],
                metadata={
                    "section": section,
                    "zone": zone,
                    "page": page_number,
                    "page_label": reader.page_labels[page_number],
                    "total_pages": len(reader.pages),
                    "source": "TENERIFE.pdf"
                }
            )
            docs.append(doc)

docs


[Document(metadata={'section': 'SITIOS QUE VER', 'zone': 'ZONA NORTE', 'page': 0, 'page_label': '1', 'total_pages': 25, 'source': 'TENERIFE.pdf'}, page_content='• Santa Cruz de Tenerife: \nSanta Cruz de Tenerife es la capital de la isla. Quizás la ruta a seguir si vais a Santa \nCruz sería: \n- Aparcar en el aparcamiento del Parque Marítimo (ubicación). \n- Caminar por la Avenida Marítima hasta Plaza de España (ubicación). \n- Por el camino de la Avenida Marítima, ver el auditorio de Tenerife (ubicación). \n- Una vez llegados a Plaza España, callejear un poco (subir la Calle Castillo \ndirección Plaza Weyler - ubicación –; ir hacia el Parque García Sanabria - \nubicación -; y bajar de nuevo hacia Plaza de España pasando por la Plaza del \nPríncipe - ubicación). \n- Volver de nuevo a por el coche. \n- Ir hacia la playa de Las Teresitas (ubicación). \nLos puntos de interés sugeridos son estos: \no Parque Marítimo César Manrique [vídeo - ubicación] \nConjunto de piscinas naturales.o Audit

# Chunking
Utizamos una estrategia de chunking cortando el texto cada 500 caracteres primero intenta cortar por párrafos \n\n, si sigue siendo muy largo corta por saltos de línea \n, luego por espacios, y como último recurso por caracteres individuales. Para el próximo recorte va 80 caracteres más atrás y por si había recortado la información y había perdido contexto.

In [47]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
    add_start_index=True,
)

chunks = text_splitter.split_documents(docs)

for i, doc in enumerate(chunks):
    doc.metadata["chunk_id"] = i
    doc.metadata["source_name"] = PDF_PATH.name

print("Número de chunks:", len(chunks))
print("\nEjemplo de chunk:\n")
print(chunks[0].page_content)
print("\nMetadata:", chunks[0].metadata)

Número de chunks: 44

Ejemplo de chunk:

• Santa Cruz de Tenerife: 
Santa Cruz de Tenerife es la capital de la isla. Quizás la ruta a seguir si vais a Santa 
Cruz sería: 
- Aparcar en el aparcamiento del Parque Marítimo (ubicación). 
- Caminar por la Avenida Marítima hasta Plaza de España (ubicación). 
- Por el camino de la Avenida Marítima, ver el auditorio de Tenerife (ubicación). 
- Una vez llegados a Plaza España, callejear un poco (subir la Calle Castillo 
dirección Plaza Weyler - ubicación –; ir hacia el Parque García Sanabria -

Metadata: {'section': 'SITIOS QUE VER', 'zone': 'ZONA NORTE', 'page': 0, 'page_label': '1', 'total_pages': 25, 'source': 'TENERIFE.pdf', 'start_index': 0, 'chunk_id': 0, 'source_name': 'TENERIFE.pdf'}


# Embeddings
Utilizaremos InMemoryVectorStore para guardar en memoria nuestros vectores para esta práctica. Con esto lo que hacemos es indexar los chunks y es más fácil para el LLM encontrar ese chunk que podría contener información relevante para poder contestar a lo que se le pregunta.

In [48]:
vector_store = InMemoryVectorStore(embeddings)
document_ids = vector_store.add_documents(chunks)

print("Documentos indexados:", len(document_ids))
print("Primeros IDs:", document_ids[:3])

Documentos indexados: 44
Primeros IDs: ['c83db7b4-cfc1-42e5-a341-f6cdcf406760', '95bb58c7-c864-4aeb-8f9b-6844009ca518', '4bd8f683-05af-479b-be8c-dfd331f53445']


# Pruebas chunking con diferentes k
En esta parte experimento con diferentes preguntas para saber cuantos chunkings son necesarios para que el modelo pueda proporcionar buenas respuestas. Me quedo con el valor k = 5.

In [49]:
#experiment_question = "¿Que sitios ver en la zona norte?"
#experiment_question = "¿Donde comer en el norte?"
experiment_question = "¿Que sitios ver en la zona sur?"

zone = None
section = None
if "norte" in experiment_question.lower():
    zone = "ZONA NORTE"
    section = "SITIOS QUE VER"
elif "sur" in experiment_question.lower():
    zone = "ZONA SUR"
    section = "SITIOS QUE VER"
if "comer" in experiment_question.lower() or "restaurante" in experiment_question.lower():
    zone = None
    section = "SITIOS PARA COMER"

def show_documents(documentos):
    for i, doc in enumerate(documentos, start=1):
        print(f"Resultado {i}")
        print("Fuente:", doc.metadata.get("source_name"))
        print("Página:", doc.metadata.get("page"))
        print("Chunk:", doc.metadata.get("chunk_id"))
        print(doc.page_content)
        print("-" * 80)

for k in [1, 2, 4, 5]:
    print(f"\n\nK = {k}")
    print("=" * 80)
    docs_k = vector_store.similarity_search(experiment_question, k=k, filter=lambda doc: (zone is None or doc.metadata.get("zone") == zone) and (section is None or doc.metadata.get("section") == section))
    show_documents(docs_k)



K = 1
Resultado 1
Fuente: TENERIFE.pdf
Página: 20
Chunk: 31
• Los Gigantes 
Los Gigantes es una zona del Sur comúnmente conocida por el Acantilado de Los 
Gigantes y donde se hacen las típicas excursiones para ver delfines y ballenas 
(recomendado). También podéis hacer excursiones en kayak, motos de agua y esas 
vainas. Para consultar información sobre este tipo de excursiones, visitad este enlace 
o buscad vosotros mismos que seguro que encontráis un montón de actividades al 
llegar a Los Gigantes.
--------------------------------------------------------------------------------


K = 2
Resultado 1
Fuente: TENERIFE.pdf
Página: 20
Chunk: 31
• Los Gigantes 
Los Gigantes es una zona del Sur comúnmente conocida por el Acantilado de Los 
Gigantes y donde se hacen las típicas excursiones para ver delfines y ballenas 
(recomendado). También podéis hacer excursiones en kayak, motos de agua y esas 
vainas. Para consultar información sobre este tipo de excursiones, visitad este enlace 
o busc

# Function tool
Se crea una function tool para poder chequear el tiempo, ya que la documentación no tiene este tipo de información. Se opta por utlizar el decorador tool, ya que, facilita la creación de la herramienta y queda más limpio en código. Luego se le inyecta la herramienta al LLM que posteriormente se utilizará.

In [63]:
def geocode_location(location: str) -> dict[str, Any]:
    """Obtiene coordenadas aproximadas para una ubicación usando Open-Meteo Geocoding."""
    response = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": location, "count": 1, "language": "es", "format": "json"},
        timeout=10,
    )
    response.raise_for_status()
    payload = response.json()
    results = payload.get("results") or []
    if not results:
        raise ValueError(f"No se encontraron coordenadas para: {location}")

    first = results[0]
    return {
        "name": first.get("name"),
        "country": first.get("country"),
        "latitude": first["latitude"],
        "longitude": first["longitude"],
        "timezone": first.get("timezone"),
    }

@tool
def get_current_weather(location: str, units: Literal["celsius", "fahrenheit"] = "celsius") -> dict[str, Any]:
    """Consulta el tiempo actual, temperatura, humedad y viento para una ciudad o municipio determinado (ejemplo: 'Santa Cruz de Tenerife', 'La Orotava')."""
    place = geocode_location(location)
    temperature_unit = "fahrenheit" if units == "fahrenheit" else "celsius"

    response = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": place["latitude"],
            "longitude": place["longitude"],
            "current": "temperature_2m,relative_humidity_2m,wind_speed_10m",
            "temperature_unit": temperature_unit,
            "timezone": "auto",
        },
        timeout=10,
    )
    response.raise_for_status()
    payload = response.json()
    current = payload.get("current") or {}
    units_payload = payload.get("current_units") or {}

    return {
        "location": place,
        "observed_at": current.get("time"),
        "temperature": current.get("temperature_2m"),
        "temperature_unit": units_payload.get("temperature_2m"),
        "relative_humidity": current.get("relative_humidity_2m"),
        "relative_humidity_unit": units_payload.get("relative_humidity_2m"),
        "wind_speed": current.get("wind_speed_10m"),
        "wind_speed_unit": units_payload.get("wind_speed_10m"),
        "provider": "Open-Meteo",
    }

tools = [get_current_weather]
llm_with_tools = llm.bind_tools(tools)

# Respuesta del modelo utilizando el RAG
Se le especifica al modelo que es un guía turístico de Tenerife y que tiene que dar las respuestas en español y no inventarse nada que no encuentre en los documentos. También tiene que citar de donde sale la iformación. Se utiliza LangGraph porque nos permite modelar el RAG como un pequeño grafo con dos nodos. De esta forma, queda más recogida la funcionalidad y se puede meter más lógica a los pasos. Se utiliza también la memoria MemorySaver para poder tener diálogo multiturno y que el llm recuerde de lo que estaba hablando. Se meten filtros en la búsqueda de los chunks a través de los metadatos que introducimos de esa forma es más fácil determinar que chunks son relevantes. De la misma forma, quitamos el contexto cuando en la pregunta leemos algo referente tiempo o clima para que en vez de ir a buscar esa información a los documentos utilice la herramienta para ello. Lo forzamos así para asegurar que la llamada se realizará porque en los documetos podría haber información relacionada con el tiempo que no nos aportaría nada.

In [ ]:
class RAGState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    context: List[Document]
    answer: str
    sources: List[dict]

rag_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Eres un guía turístico experto, muy cercano y amigable en la isla de Tenerife. "
            "Habla de forma natural, fluida y conversacional, como si estuvieras recomendándole los sitios a un amigo en persona. "
            "No suenes robótico ni hagas listados excesivamente mecánicos. "
            "Responde en español usando **únicamente** el contexto proporcionado. "
            "Si el usuario pregunta por el tiempo, el clima actual o la temperatura, tienes a tu disposición la herramienta get_current_weather para decidir usarla de forma autónoma. "
            "Si el contexto no contiene la respuesta, dilo de forma natural (por ejemplo: 'Oye, la verdad es que no tengo esa información por aquí'). "
            "No inventes datos bajo ningún concepto y añade las fuentes de forma discreta al final.",
        ),
        MessagesPlaceholder("messages"),
        (
            "user",
            "Contexto recuperado:\n{context}",
        ),
    ]
)

def retrieve(state: RAGState):
    last_message = [m for m in state["messages"] if isinstance(m, HumanMessage)][-1]
    question = last_message.content if hasattr(last_message, "content") else str(last_message)
    if any(key in question.lower() for key in ["tiempo", "clima", "temperatura", "hace"]):
        return {"context": []}
    zone = None
    section = None
    if "norte" in question.lower():
        zone = "ZONA NORTE"
        section = "SITIOS QUE VER"
    elif "sur" in question.lower():
        zone = "ZONA SUR"
        section = "SITIOS QUE VER"
    if "comer" in question.lower() or "restaurante" in question.lower():
        zone = None
        section = "SITIOS PARA COMER"
    retrieved_docs = vector_store.similarity_search(question, k=5, filter=lambda doc: (zone is None or doc.metadata.get("zone") == zone) and (section is None or doc.metadata.get("section") == section))
    return {"context": retrieved_docs}


def generate(state: RAGState):
    context = state["context"] if state["context"] else "No hay contexto de documento para esta consulta (se requiere uso de herramienta externa si aplica)."
    messages = rag_prompt.invoke({"messages": state["messages"], "context": context})
    response = llm_with_tools.invoke(messages)
    sources = [
        {
            "section": doc.metadata.get("section"),
            "zone": doc.metadata.get("zone"),
            "source": doc.metadata.get("source_name"),
            "page": doc.metadata.get("page"),
            "chunk_id": doc.metadata.get("chunk_id"),
        }
        for doc in state["context"]
    ]
    return {
        "answer": response.content if response.content else "Utilizando herramienta de clima para obtener la respuesta...", 
        "sources": sources, 
        "messages": [response]
    }

memory = MemorySaver()

tool_node = ToolNode(tools=tools)

graph_builder = StateGraph(state_schema=RAGState)
graph_builder.add_node("retrieve", retrieve)
graph_builder.add_node("generate", generate)
graph_builder.add_node("tools", tool_node)
graph_builder.add_edge(START, "retrieve")
graph_builder.add_edge("retrieve", "generate")
graph_builder.add_conditional_edges("generate", tools_condition)
graph_builder.add_edge("tools", "generate")
rag_graph = graph_builder.compile(checkpointer=memory)

test_questions = [
    "¿Que sitios ver en la zona norte?",
    "¿Y para comer?",
    "¿Estan buenos?",
    "Algún plato",
    "¿Y en el sur?",
    "¿Que tiempo hace en Santa Cruz?",
    "¿Sitios obligatorios?",
    "¿Que playas visitar?",
    "¿Que ver en Orotava?"
]

config = {"configurable": {"thread_id": 1}}

for question in test_questions:
    print("PREGUNTA:", question)
    result = rag_graph.invoke({"messages": [HumanMessage(content=question)]}, config)
    last_message = result["messages"][-1]
    response = last_message.content if getattr(last_message, "content", "") else result.get("answer", "")
    print("Respuesta:\n", response)
    if result["sources"]:
        print("\nFuentes:")
        for source in result["sources"]:
            print(source)
    print("\n")

PREGUNTA: ¿Que sitios ver en la zona norte?
Respuesta:
 ¡Perfecto, te cuento lo mejor del Norte como si te lo recomendara en persona!

Primero, los imprescindibles por zonas (todo según la guía local que tengo aquí):

- La Laguna: no te pierdas el centro histórico y la Plaza de la Iglesia de la Concepción —es perfecto para pasear y empaparte de ambiente.

- Santa Cruz: Plaza de España, el Parque García Sanabria y el Auditorio de Tenerife son visitas muy fáciles y agradables. Si te apetece playa urbana, Playa de Las Teresitas está cerca y merece la pena.

- Puerto de la Cruz: aquí tienes varias cosas en un mismo paseo: la ruta por Santo Domingo, la calle de San Telmo y la avenida Cristóbal Colón, pasando por la Plaza de Europa y el Lago/Lago de Martiánez (conjunto de piscinas naturales). La Playa Martiánez queda bien para un descanso; cerca está también el famoso Loro Parque (suele ser un “must” turístico).

- Rincón foodie y helados: si os entra hambre en la zona norte, una recomendaci

# Conclusiones finales

He hecho lo que he podido con el tiempo disponible que he tenido en las últimas semanas. Donde he tenido quebraderos de cabeza ha sido en la forma de hacer el RAG con los chunks. Le di muchas vueltas a la hora de escoger una estrategia para recortar el documento. Al final opte por secciones/zonas/lugares. 
También elegí hacerlo con LangGraph porque me parecía más estructurado pero al mismo tiempo me di cuenta de que era más complejo. El problema aquí es que me di de bruces durante unas horas con la function tool y el RAG. 
Intenté utilizar el streaming pero no supe como lograrlo con LangGraph ya que la invocación esta dentro del generate, creo que desde ahí se tiene que propagar pero no lo conseguí.
Dónde también estuve un rato largo fue haciendo el diálogo multiturno. Al final encontre como utilizar los messages porque al principio solo pasaba la pregunta a pelo.

Se que se tendría que haber estudiado más modelos y hacer más pruebas con preguntas o intentar sacar los k(s) óptimos para recuperar los chunks pero lo que comentaba era todo el tiempo que tenía disponible.